# 09 - Final Training (End-to-End)

Run the complete pipeline through `training.py`. This notebook documents
the orchestration; the actual execution happens by running
`python training.py` from the project root.


## Pipeline steps

1. Setup logging + load YAML configs
2. Ingest raw CSVs
3. Validate against schema.yaml
4. Train/test split (80/10/10)
5. Transform (identity pipeline - data is already ternary)
6. Feature engineering (30 -> 66 features)
7. Feature selection (66 -> 41 features)
8. Unsupervised clustering (10 algorithms, optional/interpretive)
9. Supervised classification (9 base + 2 ensembles + AutoGluon)
10. Evaluation (Accuracy/Precision/Recall/F1/AUC/MCC/LogLoss)
11. SHAP global explainability
12. Save best model + metadata


In [ ]:
# Equivalent to running `python training.py` from the project root.
import subprocess, sys, os
os.chdir('..')
result = subprocess.run([sys.executable, 'training.py'],
                        capture_output=True, text=True, timeout=1800)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])


## Verify artifacts

In [ ]:
import os
from pathlib import Path
base = Path('.')
for sub in ['artifacts/models/trained_models',
            'artifacts/features', 'artifacts/clusters',
            'artifacts/evaluations', 'artifacts/shap']:
    files = sorted((base / sub).glob('*'))
    print(f"\n{sub}/  ({len(files)} files)")
    for f in files[:5]:
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")


## Best model metadata

In [ ]:
import json
from src.utils import get_config
cfg = get_config()
with open(cfg.trained_models_dir / "best_model_metadata.json") as f:
    meta = json.load(f)
print(f"Best model: {meta['name']}")
print(f"Metrics:")
for k, v in meta['metrics'].items():
    if isinstance(v, (int, float)):
        print(f"  {k:20s}: {v:.4f}")
    else:
        print(f"  {k:20s}: {v}")


## Summary

After running `training.py`, all artifacts are persisted under
`artifacts/`. The best model is selected by recall on the phishing class
and saved as `best_model.joblib`, with metadata in
`best_model_metadata.json`. The Streamlit app (`app.py`) reads these
artifacts at launch time - no further training is needed for inference.
